# PlantVillage ANN（Kaggle 对比实验）

默认使用 **torchvision 官方 `resnet18`**（替换 `fc` 为类别数），与 **SNN 侧 `sj_resnet18`** 同属标准 ResNet-18 族，便于论文对比。输入 **`[B,3,H,W]`**，无时间步、无 SpikingJelly。

将下方 **`BACKBONE_ANN = "res2net"`** 可切回仓库内 **Res2Net-29 风格 ANN**（与旧版 notebook 一致）。

1. **Add Data**：PlantVillage（ImageFolder 结构）。
2. **代码**：git clone 本仓库或上传压缩包。
3. 修改 **GitHub URL** 与 **INPUT_NAME**，从上到下运行。

数据管线与 `kaggle_plantvillage_snn.ipynb` 相同：`data/kaggle_dataloader.py`；**EPOCHS / BATCH_SIZE / LR / label_smoothing / warmup** 等建议与 SNN 单元格对齐。

In [ ]:
# --- 依赖（无需 spikingjelly） ---
!pip -q install tensorboard scikit-learn

In [ ]:
# --- 拉取 GitHub 代码（把 URL 换成你的仓库） ---
import os

os.chdir('/kaggle/working')

GITHUB_URL = "https://github.com/xing11234/plantvillage_snn.git"  # 修改
BRANCH = "master"  # 或 master
CLONE_DIR = "/kaggle/working/plantvillage_snn"


!git clone --depth 1 --branch {BRANCH} {GITHUB_URL} {CLONE_DIR}

os.chdir(CLONE_DIR)


print("Working dir:", os.getcwd())

In [ ]:
# --- 若使用 Upload Code 数据集：取消注释 ---
# import os
# os.chdir("/kaggle/input/your-code-dataset/plantvillage_snn")
# print(os.getcwd())

In [ ]:
# --- Kaggle 数据集路径：INPUT 名称与 Data 面板一致 ---
import os

INPUT_NAME = "/kaggle/input/datasets/abdallahalidev/plantvillage-dataset"  # 修改：与 /kaggle/input 下文件夹名一致
DATA_ROOT = os.path.join("/kaggle/input", INPUT_NAME)

# 若类文件夹不在 INPUT 根下，保持 True 会自动搜索（如 .../color）
#AUTO_FIND_SUBDIR = True

assert os.path.isdir(DATA_ROOT), f"Missing dataset path: {DATA_ROOT}"
print("DATA_ROOT:", DATA_ROOT)

In [ ]:
# --- 训练 ANN（与 SNN 相同划分 / 增强 / 优化器超参） ---
import os
import sys

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

import torch
import torch.nn as nn
from torch import amp as torch_amp
from torch.utils.tensorboard import SummaryWriter

from config import get_config
from data.kaggle_dataloader import get_kaggle_dataloaders
from train_ann import evaluate_ann, train_one_epoch_ann
from utils.lr_schedule import cosine_lr
from utils.seed import set_seed

# "tv_resnet18" = torchvision.models.resnet18；"res2net" = 仓库 Res2Net-ANN
BACKBONE_ANN = "tv_resnet18"
# ImageNet 预训练头再微调；与「从零训练」SNN 对比时建议 False
IMAGENET_PRETRAINED = False

PRESET = None
EPOCHS = 20
BATCH_SIZE = 64
LR = 0.05
# ANN 输入为 [B,C,H,W]，dim0 是 batch，可用 DataParallel
USE_DATA_PARALLEL = torch.cuda.device_count() > 1

cfg = get_config(PRESET, epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR)
cfg.label_smoothing = 0.05  # 与 SNN 公平对比时建议一致；不需要可改 0
cfg.warmup_epochs = min(cfg.warmup_epochs, max(1, cfg.epochs // 2))
cfg.num_workers = 2
cfg.batch_log_interval = 50
cfg.spike_counter_enabled = False
cfg.save_name = (
    "best_ann_tv_resnet18.pt"
    if BACKBONE_ANN == "tv_resnet18"
    else "best_ann_res2net.pt"
)
cfg.checkpoint_dir = "/kaggle/working/checkpoints_ann"
cfg.log_dir = "/kaggle/working/runs_ann"
os.makedirs(cfg.checkpoint_dir, exist_ok=True)

set_seed(cfg.seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print(
    f"ANN backbone={BACKBONE_ANN} | warmup_epochs={cfg.warmup_epochs} | save={cfg.save_name} | "
    f"DataParallel={USE_DATA_PARALLEL and torch.cuda.device_count() > 1}",
    flush=True,
)

if "AUTO_FIND_SUBDIR" not in globals():
    AUTO_FIND_SUBDIR = True

train_loader, val_loader, num_classes = get_kaggle_dataloaders(
    DATA_ROOT,
    image_size=cfg.image_size,
    batch_size=cfg.batch_size,
    num_workers=cfg.num_workers,
    val_ratio=0.2,
    seed=cfg.seed,
    auto_find_subdir=AUTO_FIND_SUBDIR,
)
cfg.num_classes = num_classes
print("num_classes:", num_classes, "train batches:", len(train_loader), "val batches:", len(val_loader))

if BACKBONE_ANN == "tv_resnet18":
    from models.resnet18_tv_ann import build_tv_resnet18_ann_from_config

    model = build_tv_resnet18_ann_from_config(cfg, imagenet_pretrained=IMAGENET_PRETRAINED).to(
        device
    )
    print("Model: torchvision resnet18 (fc -> num_classes)", flush=True)
else:
    from models.res2net_ann import build_ann_model

    model = build_ann_model(cfg).to(device)
    print("Model: Res2NetANN (仓库与 MSF-Res2Net 对齐的 ANN)", flush=True)
if USE_DATA_PARALLEL and torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
    print(f"nn.DataParallel on {torch.cuda.device_count()} GPUs", flush=True)


def _unwrap(m):
    return m.module if isinstance(m, nn.DataParallel) else m


optimizer = torch.optim.SGD(
    model.parameters(),
    lr=cfg.lr,
    momentum=cfg.momentum,
    weight_decay=cfg.weight_decay,
    nesterov=True,
)
scaler = torch_amp.GradScaler(enabled=cfg.amp and device.type == "cuda")
writer = SummaryWriter(log_dir=os.path.join(cfg.log_dir, "kaggle"))

print(
    f"Starting training: {cfg.epochs} epochs, {len(train_loader)} train batches / epoch",
    flush=True,
)
best_acc = 0.0
for epoch in range(cfg.epochs):
    lr_now = cosine_lr(epoch, cfg.lr, cfg.warmup_epochs, cfg.epochs, cfg.min_lr)
    for pg in optimizer.param_groups:
        pg["lr"] = lr_now

    print(f"\n--- Epoch {epoch + 1}/{cfg.epochs}  lr={lr_now:.5f} ---", flush=True)
    train_loss, train_acc = train_one_epoch_ann(
        model, train_loader, optimizer, scaler, device, cfg
    )
    print("  validating...", flush=True)
    val_acc = evaluate_ann(model, val_loader, device, cfg.amp)

    writer.add_scalar("loss/train", train_loss, epoch)
    writer.add_scalar("acc/train", train_acc, epoch)
    writer.add_scalar("acc/val", val_acc, epoch)

    print(
        f"Epoch {epoch+1}/{cfg.epochs} lr={lr_now:.5f} loss={train_loss:.4f} "
        f"train_acc={train_acc*100:.2f}% val_acc={val_acc*100:.2f}%"
    )

    if val_acc > best_acc:
        best_acc = val_acc
        ckpt_path = os.path.join(cfg.checkpoint_dir, cfg.save_name)
        torch.save(
            {
                "model": _unwrap(model).state_dict(),
                "cfg": cfg.to_dict(),
                "epoch": epoch,
                "val_acc": val_acc,
                "backbone": (
                    "torchvision_resnet18"
                    if BACKBONE_ANN == "tv_resnet18"
                    else "Res2NetANN"
                ),
            },
            ckpt_path,
        )
        print("  saved", ckpt_path)

writer.close()
print(f"Done. Best val acc = {best_acc*100:.2f}%")

## 输出

- 权重：`/kaggle/working/checkpoints_ann/best_ann_tv_resnet18.pt`（`BACKBONE_ANN="tv_resnet18"`）或 `best_ann_res2net.pt`
- 日志：`/kaggle/working/runs_ann`，本地 `tensorboard --logdir runs_ann`